Imports

In [22]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [23]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=1:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

In [24]:
cluster.scale(jobs=1)

In [15]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [26]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat,retain_design_matricies=True)
test.extract_params(client)

In [27]:
cell_counts=scm.get_cell_counts(client,dat,split="cre_id")

flattened_param=scm.flatten_param_representation(client,test.by_cre_parameters.result(),split="cre_id")

In [34]:
working=cell_counts.join(flattened_param)
working["r"]=np.exp(working["theta"])
working["sigmasquare"]=working["nb"]**2/working["r"]+working["nb"]
working

cells  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id             
1                   0                   1              0              1            0            0            nobody       503   
                                                                      0            1            0            nobody       503   
                                                                                   0            1            nobody       503   
                                                                      1            0            0            somebody     503   
                                                                      0            1            0            somebody     503   
...                                                                                                                       ...   
0                   1                   0              1              0            1            0            redgene      453   
                                                                                   0            1            redgene      453   
                                                                      1            0            0            neurogene    453   
                                                                      0            1            0            neurogene    453   
                                                                                   0            1            neurogene    453   

                                                                                                                           cre_id  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id                 
1                   0                   1              0              1            0            0            nobody        nobody   
                                                                      0            1            0            nobody        nobody   
                                                                                   0            1            nobody        nobody   
                                                                      1            0            0            somebody      nobody   
                                                                      0            1            0            somebody      nobody   
...                                                                                                                           ...   
0                   1                   0              1              0            1            0            redgene    neurogene   
                                                                                   0            1            redgene    neurogene   
                                                                      1            0            0            neurogene  neurogene   
                                                                      0            1            0            neurogene  neurogene   
                                                                                   0            1            neurogene  neurogene   

                                                                                                                               nb  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id                 
1                   0                   1              0              1            0            0            nobody      1.770551   
                                                                      0            1            0            nobody      1.770551   
                                                                                   0            1            nobody      1.770551   
                                               

In [21]:
cluster.close()